<a href="https://colab.research.google.com/github/muyuanlidvm/ansci_4040_final_class_project/blob/ParlorTest/ANSC4040ParlorIDMatch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assign IDs to cows entering parlor

The goal of this notebook is to create tracking ID for each cow entering the parlor. The tracking IDs will be ordered in sequence of first appearance. Based on the created order, cows will be matched with the list of cows pulled from BoviSync record. Cows from the record were ordered by time and parlor stall number.

The video used as an example is Entry Parlor video 10.

Below are some assumptions made:
Bovisync list corresponds to

*   Bovisync list corresponds to October 3rd parlor video.
*   Beginning of milking shift starts at 3 AM; the first cow that enters stall one is the first cow that enters the parlor in the video.
*   Left side of parlor is 1-28 (cows entered first in the video), left side is 29-56.
*   We will ignore cows with no stall label.



In [1]:
pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 11.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [ultralytics]


In [2]:
from ultralytics import YOLO
import cv2, os, json
import numpy as np
import pandas as pd


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# Set working directory
%cd /content/drive/Shareddrives/BeastsAndBytes/ANSC 4040 Group Project/ParlorIDMatch


/content/drive/Shareddrives/BeastsAndBytes/ANSC 4040 Group Project/ParlorIDMatch


In [7]:
model = YOLO("yolo26m-seg.pt")  # or your custom weights
video_path = "/content/drive/Shareddrives/BeastsAndBytes/ANSC 4040 Group Project/Entry Parlor Videos/2026-10-03 00-00-00 Entry  Parlor (10).avi"
out_dir = "cow_segments"
os.makedirs(out_dir, exist_ok=True)

In [8]:
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)

In [10]:
records = []   # one row per (track, frame)
frame_idx = 0

results = model.track(video_path, stream=True, persist=True,
                      tracker="bytetrack.yaml", classes=[19], # 19 = cow in COCO
                      conf=0.4)

In [14]:
MAX_SECONDS = 10 * 60 # Example - just looking at the first 10 minutes

for r in results:
    t = frame_idx / fps
    if r.boxes.id is not None:
        ids = r.boxes.id.int().cpu().tolist()
        boxes = r.boxes.xyxy.cpu().numpy()
        masks = r.masks.data.cpu().numpy()  # (N, H, W)
        for tid, box, m in zip(ids, boxes, masks):
            records.append({
                "track_id": tid,
                "frame": frame_idx,
                "time_s": round(t, 3),
                "x1": float(box[0]), "y1": float(box[1]),
                "x2": float(box[2]), "y2": float(box[3]),
                "cx": float((box[0] + box[2]) / 2),
            })
            # save the segment (mask applied to the frame)
            tdir = os.path.join(out_dir, f"track_{tid:04d}")
            os.makedirs(tdir, exist_ok=True)
            mask = cv2.resize(m, (r.orig_img.shape[1], r.orig_img.shape[0])) > 0.5
            seg = r.orig_img.copy()
            seg[~mask] = 0
            x1, y1, x2, y2 = map(int, box)
            cv2.imwrite(os.path.join(tdir, f"f{frame_idx:06d}_t{t:.2f}.png"),
                        seg[y1:y2, x1:x2])
    frame_idx += 1

In [15]:
df = pd.DataFrame(records)

#line crossing (cows moving left->right across x=900)
LINE_Y = 80
crossed = df[df["y1"] <= LINE_Y]
order = (crossed.groupby("track_id")["time_s"].min()
                .sort_values().reset_index()
                .rename(columns={"time_s": "entry_time_s"}))
order["entry_rank"] = range(1, len(order) + 1)


In [17]:
# Read a frame from the video to visualize the line crossing boundary
# (Re-initialize video capture to ensure we read from the beginning if needed)
cap_check = cv2.VideoCapture(video_path)
ret, frame = cap_check.read()

if ret:
    h, w, c = frame.shape
    # Draw a horizontal line at LINE_Y (80)
    cv2.line(frame, (0, 80), (w, 80), (0, 0, 255), 2)
    cv2.imwrite("check_line.png", frame)
    print("Successfully saved 'check_line.png'")
else:
    print("Could not read frame from the video path.")

cap_check.release()

Successfully saved 'check_line.png'


In [25]:
# Match ID by ranks
xl = pd.read_excel("Nov1and2MilkingDataOrdered_new.xlsx")   # columns e.g. stall, cow_id
xl["entry_rank"] = xl["RankFix"]       # stall 1 = 1st cow in

matched = order.merge(xl, on="entry_rank", how="outer")
matched.to_csv("track_to_cow.csv", index=False)